CSU Channel Island

In [ ]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup


def first_last_name(name):
    """Turns 'Abbasi, Bahareh' into 'Bahareh Abbasi'."""
    if "," in name:
        last, first = name.split(",", 1)
        return f"{first.strip()} {last.strip()}"

    return name.strip()


def get_department(title):
    """
    Examples:
    'Associate Professor - Mechatronics Engineering'
    -> 'Mechatronics Engineering'

    'Lecturer AY - Anthropology - 3'
    -> 'Anthropology'
    """
    if " - " in title:
        parts = title.split(" - ")

        # The department is normally the part after the job title.
        return parts[0].strip(), parts[1].strip()

    # Handles titles such as "Professor of Chemistry"
    match = re.search(
        r"(?:Professor|Lecturer|Instructor)\s+(?:of|in)\s+(.+)",
        title,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(0).strip(), match.group(1).strip()

    return None, None

faculty = []

for i in range(1, 6):
  print(i)
  url = f"https://ciapps.csuci.edu/directory/Home?id=Faculty&filters=Faculty&optionsVisible=true&page={i}"

  response = requests.get(
      url,
      headers={"User-Agent": "Mozilla/5.0"}
  )
  response.raise_for_status()

  soup = BeautifulSoup(response.text, "html.parser")



  for row in soup.find_all("tr"):
      name_cell = row.find("th", scope="row")
      cells = row.find_all("td")

      if name_cell is None or len(cells) < 1:
          continue

      name_link = name_cell.find("a")

      if name_link is None:
          continue

      original_name = name_link.get_text(" ", strip=True)
      title = cells[0].get_text(" ", strip=True)

      title, department = get_department(title)

      faculty.append({
          "name": first_last_name(original_name),
          "department": department,
          "title": title,
          "school": "CSU Channel Islands",
          "major": "Null"
      })

df = pd.DataFrame(faculty)

print(df.head())
print("Faculty found:", len(df))

df.to_parquet("csuci_faculty.parquet", index=False)

1
2
3
4
5
                       name                department                title  \
0            Bahareh Abbasi  Mechatronics Engineering  Associate Professor   
1              Reza Abdolee                  Comp Sci  Associate Professor   
2        Leslie Marie Abell                      None                 None   
3   Marc Alexander Abramiuk              Anthropology          Lecturer AY   
4  Christopher James Acosta      Academic Internships         Faculty Lead   

                school major  
0  CSU Channel Islands  Null  
1  CSU Channel Islands  Null  
2  CSU Channel Islands  Null  
3  CSU Channel Islands  Null  
4  CSU Channel Islands  Null  
Faculty found: 461


CSU Bakersfield

In [10]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup


def first_last_name(name):
    """Turns 'Acharya, Tathagata' into 'Tathagata Acharya'."""
    if "," in name:
        last, first = name.split(",", 1)
        return f"{first.strip()} {last.strip()}"

    return name.strip()


def get_department(title):
    """
    Example:
    'Chair and Associate Professor of Chemistry and Biochemistry'
    -> 'Chemistry and Biochemistry'
    """
    match = re.search(
        r"(?:Professor|Lecturer|Instructor|Professor Emeritus|Lecturer Emeritus)\s+(?:of|in)\s+(.+)",
        title,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(0).split(" of ")[0].strip(), match.group(1).strip()

    # Some records may not clearly state a department
    return None, None



import time

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(headers)

# Establish a normal catalog session first
session.get("https://catalog.csub.edu/", timeout=30)

url = "https://catalog.csub.edu/general-information/csub-information/faculty_list/"

for _ in range(5):
    response = session.get(url, timeout=30)

    if response.status_code == 200:
        break

    print("Server returned:", response.status_code, "— retrying...")
    time.sleep(2)

response.raise_for_status()
print("Rows found:", len(BeautifulSoup(response.text, "html.parser").find_all("tr")))


soup = BeautifulSoup(response.text, "html.parser")

faculty = []

for row in soup.find_all("tr"):
    cells = row.find_all("td")

    if len(cells) != 2:
        continue

    original_name = cells[0].get_text(" ", strip=True)
    title = cells[1].get_text(" ", strip=True)

    title, department = get_department(title)

    if title == None and department == None:
      continue

    faculty.append({
        "name": first_last_name(original_name),
        "department": department,
        "title": title,
        "school": "CSU Bakersfield",
    })

df = pd.DataFrame(faculty)

print(df.head())
print("Faculty found:", len(df))

df.to_parquet("csub_faculty.parquet", index=False)

Rows found: 501
                       name                        department      title  \
0         Tathagata Acharya           Physics and Engineering  Professor   
1      Moises Acuna-Gurrola                           History  Professor   
2  Mian Ahmed Shaheer Afaqi  Philosophy and Religious Studies  Professor   
3            Ankita Agarwal          Management and Marketing  Professor   
4                Andy Alali                    Communications  Professor   

            school  
0  CSU Bakersfield  
1  CSU Bakersfield  
2  CSU Bakersfield  
3  CSU Bakersfield  
4  CSU Bakersfield  
Faculty found: 466


CSU Northridge


In [14]:
!playwright install --with-deps chromium

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates/restricted amd64 Packages [2,231 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.4 MB]
Get:11 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 Packages [1,729 kB]
Get:12 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,168 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-upda

In [5]:
!pip install playwright pyarrow
!playwright install chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 19.0 MB/s eta 0:00:00
191.1 MiB [] 0% 0.0s191.1 MiB [] 0% 57.1s191.1 MiB [] 0% 61.5s191.1 MiB [] 0% 35.8s191.1 MiB [] 0% 28.8s191.1 MiB [] 0% 27.5s191.1 MiB [] 0% 23.5s191.1 MiB [] 0% 14.0s191.1 MiB [] 1% 9.7s191.1 MiB [] 1% 7.8s191.1 MiB [] 2% 6.7s191.1 MiB [] 3% 5.9s191.1 MiB [] 3% 5.5s191.1 MiB [] 4% 4.9s191.1 MiB [] 4% 5.2s191.1 MiB [] 5% 4.9s191.1 MiB [] 5% 4.7s191.1 MiB [] 6% 4.5s191.1 MiB [] 6% 4.3s191.1 MiB [] 7% 4.1s191.1 MiB [] 8% 3.7s191.1 MiB [] 9% 3.5s191.1 MiB [] 10% 3.3s191.1 MiB [] 11% 3.2s191.1 MiB [] 12% 3.1s191.1 MiB [] 13% 3.2s191.1 MiB [] 13% 3.4s191.1 MiB [] 14% 3.2s191.1 MiB [] 15% 3.1s191.1 MiB [] 16% 3.0s191.1 MiB [] 17% 2.9s191.1 MiB [] 18% 2.9s191.1 MiB [] 18% 3.0s191.1 MiB [] 19% 3.0s191.1 MiB [] 20% 3.0s191.1 MiB [] 21% 2.9s191.1 MiB [] 22% 2.9s191.1 MiB [] 22% 2.8s191.1 MiB [] 23% 2.8s191.1 MiB [] 23% 2.9s191.1 MiB [] 23

In [19]:
import re
import string
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


def first_last_name(name):
    """Turns 'Abdelsayed, Michael' into 'Michael Abdelsayed'."""
    if "," in name:
        last, first = name.split(",", 1)
        return f"{first.strip()} {last.strip()}"

    return name.strip()


def get_department(title):
    """
    Example:
    'Assistant Professor of Biology'
    -> 'Biology'
    """
    match = re.search(
        r"(?:Professor|Lecturer|Instructor)\s+(?:of|in)\s+(.+)",
        title,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(0).split(" of ")[0].strip(), match.group(1).strip()

    return None, None

async def scrape_csun():
    faculty = []
    failed_letters = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        for letter in string.ascii_lowercase:
            url = f"https://catalog.csun.edu/faculty/{letter}/"

            try:
                response = await page.goto(
                    url,
                    wait_until="domcontentloaded"
                )

                if response is None or response.status >= 400:
                    status = response.status if response else "no response"
                    print(f"Could not load {letter.upper()}: {status}")

                    failed_letters.append({
                        "letter": letter.upper(),
                        "url": url,
                        "status": status
                    })
                    continue

                soup = BeautifulSoup(
                    await page.content(),
                    "html.parser"
                )

                people_on_page = 0

                for description in soup.find_all(
                    string=re.compile(r"^\s*\(\d{4}\)")
                ):
                    bio = description.get_text(" ", strip=True)
                    name_link = description.find_previous("a")

                    if name_link is None:
                        continue

                    name = name_link.get_text(" ", strip=True)

                    title_match = re.match(
                        r"^\(\d{4}\)\s*(.*?)\.",
                        bio
                    )



                    if title_match and len(name) > 1:
                        title = title_match.group(1)
                        title, department = get_department(title)

                        if title == None and department == None:
                          continue

                        faculty.append({
                            "name": first_last_name(name),
                            "department": department,
                            "title": title,
                            "school": "CSU Northridge",
                        })

                        people_on_page += 1

                print(f"{letter.upper()}: {people_on_page} people found")

                # One-second pause between pages
                await page.wait_for_timeout(1000)

            except Exception as error:
                print(f"Error on {letter.upper()}: {error}")

                failed_letters.append({
                    "letter": letter.upper(),
                    "url": url,
                    "status": str(error)
                })

        await browser.close()

    return faculty, failed_letters

In [20]:
faculty, failed_letters = await scrape_csun()

df = (
    pd.DataFrame(faculty)
    .drop_duplicates(subset=["name", "department"])
    .sort_values("name")
)

print(df.head())
print("Faculty found:", len(df))

df.to_parquet("csun_faculty.parquet", index=False)

A: 33 people found
B: 59 people found
C: 56 people found
D: 34 people found
E: 16 people found
F: 23 people found
G: 57 people found
H: 60 people found
I: 3 people found
J: 25 people found
K: 42 people found
L: 44 people found
M: 77 people found
N: 18 people found
O: 13 people found
P: 31 people found
Could not load Q: 429
Could not load R: 429
Could not load S: 429
Could not load T: 429
Could not load U: 429
Could not load V: 429
Could not load W: 429
Could not load X: 429
Could not load Y: 429
Could not load Z: 429
                    name                  department      title  \
150           Aaron Daly              Art and Design  Professor   
75   Abdelaziz Boulesbaa  Chemistry and Biochemistry  Professor   
215          Abel Franco                  Philosophy  Professor   
526    Abhijit Mukherjee      Mechanical Engineering  Professor   
371       Adam B. Kaplan            Computer Science  Professor   

             school  
150  CSU Northridge  
75   CSU Northridge  
215  CSU

CSU Los Angeles


In [21]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

faculty = []

for i in range(1,13):
  url = f"https://www.calstatela.edu/facultydirectory?page={i}"
  response = requests.get(
      url,
      headers={"User-Agent": "Mozilla/5.0"}
  )
  response.raise_for_status()

  soup = BeautifulSoup(response.text, "html.parser")



  for row in soup.find_all("tr"):
      cells = row.find_all("td")

      # Faculty rows have: name, department, email
      if len(cells) < 2:
          continue

      name = cells[0].get_text(" ", strip=True)
      department = cells[1].get_text(" ", strip=True)

      # Ignore the table header or blank rows
      if not name or name == "Name":
          continue

      if department == None:
        continue

      faculty.append({
          "name": name,
          "department": department,
          "school": "CSU Los Angeles"
      })

df = pd.DataFrame(faculty).drop_duplicates(subset=["name"])

print(df.head())
print("Faculty found:", len(df))

df.to_parquet("csula_faculty.parquet", index=False)

                  name                                    department  \
0       Best, Sherwood  Department of Special Education & Counseling   
1         Beyer, Robbi                     Department of Kinesiology   
2       Bezdecny, Kris           Geography, Geology, and Environment   
3         Black, Sarah                            Extended Education   
4  Blaszczynski, Carol                      Department of Management   

            school  
0  CSU Los Angeles  
1  CSU Los Angeles  
2  CSU Los Angeles  
3  CSU Los Angeles  
4  CSU Los Angeles  
Faculty found: 480


CSU Long Beach


In [2]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup


def first_last_name(name):
    """Turns 'Smith, Jane' into 'Jane Smith'."""
    if "," in name:
        last, first = name.split(",", 1)
        return f"{first.strip()} {last.strip()}"

    return name.strip()
from urllib.parse import urlparse, unquote


def department_from_url(url):
    path_parts = [
        unquote(part)
        for part in urlparse(url).path.strip("/").split("/")
        if part
    ]

    if not path_parts:
        return None

    # The final URL part is usually the directory page:
    # faculty, faculty-staff, people, department-directory, etc.
    final_part = path_parts[-1].lower()

    if any(word in final_part for word in [
        "faculty",
        "staff",
        "directory",
        "people"
    ]):
        path_parts.pop()

    if not path_parts:
        return None

    department_slug = path_parts[-1]

    # Turns "school-of-social-work" into "School Of Social Work"
    department = department_slug.replace("-", " ").title()

    return department


urls = [
     "https://sso.csulb.edu",
 "https://www.csulb.edu/cinematic-arts/department-of-cinematic-arts-directory",
                                              "https://www.csulb.edu/school-of-art/faculty-and-staff",
                 "https://www.csulb.edu/dance/faculty-staff-directory",
             "https://web.csulb.edu/colleges/cota/music/faculty-&-staff/",
                                                     "https://www.csulb.edu/theatre-arts/faculty-staff",
          "https://www.csulb.edu/college-of-education/advanced-studies-education-and-counseling/faculty-staff",
                                  " https://www.csulb.edu/college-of-education/college-of-education-faculty-staff",
"https://www.csulb.edu/college-of-education/educational-leadership/faculty-staff",
                                        "https://www.csulb.edu/college-of-education/liberal-studies/faculty-staff",
                                                    "https://www.csulb.edu/college-of-education/teacher-education/faculty-staff-0",
                               "http://web.csulb.edu/colleges/cba/contact/index.php?dept=1",
                                      "https://web.csulb.edu/colleges/cob/contact/index.php?dept=3",
                                        "https://web.csulb.edu/colleges/cob/contact/index.php?dept=4",
                            "http://web.csulb.edu/colleges/cba/contact/index.php?dept=5",
                                   "https://web.csulb.edu/colleges/cob/contact/index.php?dept=6",
                                          "https://www.csulb.edu/hung-family-college-of-engineering/biomedical-engineering/faculty-staff",
                                "https://www.csulb.edu/hung-family-college-of-engineering/chemical-engineering/faculty-staff",
                      "https://www.csulb.edu/college-of-engineering/civil-engineering-construction-engineering-management/faculty-staff",
                         "https://www.csulb.edu/college-of-engineering/computer-engineering-computer-science/faculty-staff",
           "https://www.csulb.edu/hung-family-college-of-engineering/electrical-engineering/faculty-staff",
                             "https://www.csulb.edu/hung-family-college-of-engineering/mechanical-aerospace-engineering/mae-people",
                                            "https://www.csulb.edu/college-of-health-human-services/health-science/faculty-staff",

                                                   "https://www.csulb.edu/college-of-health-human-services/health-care-administration/faculty-staff",
                        "https://www.csulb.edu/college-of-health-human-services/school-of-social-work/faculty-staff",
                            "https://www.csulb.edu/college-of-health-human-services/kinesiology/faculty-staff-0",
                     "https://www.csulb.edu/college-of-health-human-services/school-of-nursing/faculty-and-staff",
                                                    "https://www.csulb.edu/college-of-health-human-services/speech-language-pathology/faculty-staff",
                                       "https://www.csulb.edu/college-of-health-human-services/physical-therapy/faculty-staff",
                                                  "https://www.csulb.edu/college-of-health-human-services/gerontology/faculty-staff",
                            "https://www.csulb.edu/college-of-health-human-services/child-development-family-studies/faculty",
     "https://www.csulb.edu/college-of-health-human-services/recreation-and-leisure-studies/faculty-staff",
     "https://www.csulb.edu/college-of-health-human-services/family-and-consumer-sciences/faculty-and-staff",
     "https://www.csulb.edu/college-of-health-human-services/public-policy-and-administration/faculty-staff",
     "https://www.csulb.edu/college-of-health-human-services/hospitality-management/faculty",
     "https://www.csulb.edu/college-of-education/liberal-studies/faculty-staff",
     "https://www.csulb.edu/college-of-liberal-arts/psychology/faculty-staff",
     "https://www.csulb.edu/college-of-liberal-arts/anthropology/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/philosophy/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/chicano-and-latino-studies/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/english/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/africana-studies/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/american-indian-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/religious-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/asl-linguistics-deaf-cultures-program/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/global-studies/core-faculty",
     "https://www.csulb.edu/college-of-liberal-arts/comparative-world-literature-and-classics/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/department-of-history/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/american-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/economics/department-faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/geography/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/human-development/hdev-faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/journalism-public-relations/faculty",
     "https://www.csulb.edu/school-of-art/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/asian-and-asian-american-studies/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/communication-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/environmental-science-policy/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/modern-jewish-studies/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/linguistics-department/faculty",
     "https://www.csulb.edu/college-of-liberal-arts/political-science/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/romance-german-russian-languages-and-literatures/people",
     "https://www.csulb.edu/college-of-liberal-arts/sociology/faculty-and-staff",
     "https://www.csulb.edu/college-of-liberal-arts/womens-gender-sexuality-studies/wgss-faculty-staff",
     "https://www.csulb.edu/biological-sciences/department-directory",
     "https://www.csulb.edu/chemistry-biochemistry/faculty",
     "https://www.csulb.edu/earth-science/department-directory",
     "https://www.csulb.edu/mathematics-statistics/department-directory",
     "https://www.csulb.edu/physics-astronomy/department-directory",
     "https://www.csulb.edu/science-education/department-directory",
     "https://www.csulb.edu/college-of-liberal-arts/environmental-science-policy/faculty"

]

import re
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0"})

people = []
failed_pages = []
no_directory_table = []


def clean_column_name(column):
    """Makes column names easier to compare."""
    return re.sub(r"[^a-z]", "", str(column).lower())


def find_column(columns, options):
    """
    Finds a DataFrame column whose cleaned name contains
    one of the requested options.
    """

    for column in columns:
        cleaned = clean_column_name(column)

        for option in options:
            if option in cleaned:
                return column

    return None

TITLE_WORDS = (
    "professor", "lecturer", "instructor", "chair", "dean",
    "director", "coordinator", "librarian", "assistant",
    "associate", "emeritus", "administrator"
)

def class_text(tag):
    """Returns all CSS classes on an element as one lowercase string."""
    return " ".join(tag.get("class", [])).lower()


def looks_like_name(text):
    """Loose check for names such as 'Jane Smith' or 'Smith, Jane'."""
    if not text or len(text) > 80:
        return False

    words = text.replace(",", " ").split()

    return (
        len(words) >= 2
        and len(words) <= 5
        and not any(word.lower() in {
            "faculty", "staff", "directory", "contact",
            "department", "college", "university"
        } for word in words)
    )


def scrape_face_cards(soup, url, people, seen):
    """
    Scrapes profile/card layouts with an image, a name, and optional title/major.
    """
    count = 0
    department = department_from_url(url)

    # Finds likely card containers based on their CSS class names.
    possible_cards = soup.find_all(
        lambda tag: tag.name in ["article", "div", "li"]
        and any(word in class_text(tag) for word in [
            "card", "profile", "person", "faculty", "staff", "employee"
        ])
    )

    for card in possible_cards:
        # Skip large wrapper sections that contain other cards.
        child_cards = card.find_all(
            lambda tag: tag is not card
            and tag.name in ["article", "div", "li"]
            and any(word in class_text(tag) for word in [
                "card", "profile", "person", "faculty", "staff", "employee"
            ])
        )

        if child_cards:
            continue

        # A real face card should normally have an image, email, or profile link.
        if not (
            card.find("img")
            or card.find("a", href=re.compile(r"mailto:", re.I))
        ):
            continue

        # Prefer a heading for the name.
        name = None
        heading = card.find(["h2", "h3", "h4", "h5", "h6"])

        if heading:
            name = text_or_none(heading.get_text(" ", strip=True))

        # Some cards use a link or image alt text instead of a heading.
        if not looks_like_name(name):
            image = card.find("img")
            image_alt = image.get("alt") if image else None

            if looks_like_name(image_alt):
                name = image_alt

        if not looks_like_name(name):
            continue

        lines = [
            text_or_none(line)
            for line in card.get_text("\n", strip=True).split("\n")
        ]
        lines = [line for line in lines if line]

        title = None
        major = None

        for line in lines:
            if line == name or is_email_or_contact(line):
                continue

            if title is None and looks_like_title(line):
                title = line
                continue

            # Only use an explicitly labeled specialty/major field.
            major_match = re.search(
                r"(?:major|specialization|area of expertise|research area)\s*:?\s*(.+)",
                line,
                flags=re.IGNORECASE,
            )

            if major_match:
                major = major_match.group(1).strip()

        add_person(people, seen, name, title, major, department, url)
        count += 1

    return count


def text_or_none(value):
    """Returns clean text, or None for empty / NaN values."""
    if value is None or pd.isna(value):
        return None

    value = re.sub(r"\s+", " ", str(value)).strip()
    return value if value and value.lower() != "nan" else None


def looks_like_title(text):
    """Determines whether a line is likely a job title."""
    if not text:
        return False

    text = text.lower()
    return any(word in text for word in TITLE_WORDS)


def is_email_or_contact(text):
    """Skips email, phone, office, and degree lines."""
    text = text.lower()

    return (
        "@" in text
        or "office:" in text
        or "phone:" in text
        or "ph.d" in text
        or "phd" in text
        or "m.a." in text
        or "m.s." in text
        or "b.a." in text
        or re.search(r"\(?\d{3}\)?[-\s]\d{3}[-\s]\d{4}", text)
    )


def add_person(people, seen, name, title, major, department, url):
    """Adds one person while avoiding duplicate records."""
    name = text_or_none(name)

    if not name:
        return

    name = first_last_name(name)

    # Avoid headings such as "Faculty," "Staff," and "Administration."
    if name.lower() in {
        "faculty", "staff", "faculty and staff",
        "administration", "contact", "directory"
    }:
        return

    key = (name.lower(), url)

    if key in seen:
        return

    seen.add(key)

    people.append({
        "name": name,
        "title": text_or_none(title),
        "major": text_or_none(major),
        "department": department,
        "school": "CSU Long Beach",
    })


def scrape_tables(soup, url, people, seen):
    """Extract people from directory-style HTML tables."""
    count = 0

    for table in soup.find_all("table"):
        try:
            table_df = pd.read_html(StringIO(str(table)))[0]
        except ValueError:
            continue

        table_df.columns = [
            " ".join(map(str, col)) if isinstance(col, tuple) else str(col)
            for col in table_df.columns
        ]

        first_name_col = find_column(table_df.columns, ["firstname"])
        last_name_col = find_column(table_df.columns, ["lastname"])
        name_col = find_column(
            table_df.columns,
            ["name", "faculty", "person", "employee"]
        )
        title_col = find_column(
            table_df.columns,
            ["title", "position", "rank", "role"]
        )
        major_col = find_column(
            table_df.columns,
            ["major", "specialization", "concentration", "expertise"]
        )
        department_col = find_column(
            table_df.columns,
            ["department", "area", "unit", "program"]
        )

        if name_col is None and not (first_name_col and last_name_col):
            continue

        for _, row in table_df.iterrows():
            if first_name_col and last_name_col:
                name = f"{row[first_name_col]} {row[last_name_col]}"
            else:
                name = row[name_col]

            name = text_or_none(name)

            if not name or name.lower() in {"name", "faculty", "staff"}:
                continue

            title = text_or_none(row[title_col]) if title_col else None
            major = text_or_none(row[major_col]) if major_col else None

            department = (
                text_or_none(row[department_col])
                if department_col
                else department_from_url(url)
            )

            add_person(people, seen, name, title, major, department, url)
            count += 1

    return count


def scrape_heading_profiles(soup, url, people, seen):
    """
    Extract people from pages formatted like:

    ## Last, First - Major/Specialization
    Professor
    Ph.D. ...
    email@csulb.edu
    """
    count = 0
    department = department_from_url(url)

    for heading in soup.find_all(["h2", "h3", "h4", "h5", "h6"]):
        heading_text = heading.get_text(" ", strip=True)

        # The name is usually before " - ", and the major is after it.
        if " - " in heading_text:
            name, major = heading_text.split(" - ", 1)
        else:
            name, major = heading_text, None

        name = text_or_none(name)

        # CSULB faculty headings normally use "Last, First".
        # This prevents section headings such as "Faculty" or "Staff" from being saved.
        if not name or "," not in name:
            continue

        title = None

        # Look only at content after this heading, stopping at the next heading.
        for element in heading.find_all_next():
            if element is not heading and element.name in ["h2", "h3", "h4", "h5", "h6"]:
                break

            if element.name not in ["p", "div", "span", "li", "strong"]:
                continue

            text = text_or_none(element.get_text(" ", strip=True))

            if not text or is_email_or_contact(text):
                continue

            if looks_like_title(text):
                title = text
                break

        add_person(people, seen, name, title, major, department, url)
        count += 1

    return count


people = []
failed_pages = []
no_people_found = []
seen = set()

for raw_url in urls:
    url = raw_url.strip()

    if "sso.csulb.edu" in url:
        continue

    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
    except requests.RequestException as error:
        print(f"Could not load: {url}")
        failed_pages.append({
            "source_url": url,
            "error": str(error)
        })
        continue

    soup = BeautifulSoup(response.text, "html.parser")

    table_count = scrape_tables(soup, url, people, seen)
    heading_count = scrape_heading_profiles(soup, url, people, seen)
    card_count = scrape_face_cards(soup, url, people, seen)

    page_count = table_count + heading_count + card_count


    if page_count == 0:
        no_people_found.append({"source_url": url})

    print(f"{page_count} people found: {url}")

df = pd.DataFrame(people)

df.to_parquet("csulb_faculty.parquet", index=False)
pd.DataFrame(failed_pages).to_csv("failed_pages.csv", index=False)
pd.DataFrame(no_people_found).to_csv("pages_to_review.csv", index=False)

print(df[["name", "title", "major", "department"]].head())
print("Total people found:", len(df))

46 people found: https://www.csulb.edu/cinematic-arts/department-of-cinematic-arts-directory
75 people found: https://www.csulb.edu/school-of-art/faculty-and-staff
10 people found: https://www.csulb.edu/dance/faculty-staff-directory
17 people found: https://web.csulb.edu/colleges/cota/music/faculty-&-staff/
12 people found: https://www.csulb.edu/theatre-arts/faculty-staff
28 people found: https://www.csulb.edu/college-of-education/advanced-studies-education-and-counseling/faculty-staff
117 people found: https://www.csulb.edu/college-of-education/college-of-education-faculty-staff
16 people found: https://www.csulb.edu/college-of-education/educational-leadership/faculty-staff
15 people found: https://www.csulb.edu/college-of-education/liberal-studies/faculty-staff
17 people found: https://www.csulb.edu/college-of-education/teacher-education/faculty-staff-0
34 people found: http://web.csulb.edu/colleges/cba/contact/index.php?dept=1
33 people found: https://web.csulb.edu/colleges/cob/cont

Cal Poly SLO CS department


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

URL = "https://csc.calpoly.edu/faculty/"

DEPARTMENT = "Computer Science and Software Engineering"
SCHOOL = "Cal Poly San Luis Obispo"


def clean(text):
    return " ".join(text.split())


def first_last_name(name):
    """Turns 'Abbasi, Bahareh' into 'Bahareh Abbasi'."""
    if "," in name:
        last, first = name.split(",", 1)
        return f"{first.strip()} {last.strip()}"

    return name.strip()


def scrape_csse_faculty():
    response = requests.get(
        URL,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Locate the table immediately after the "Faculty" heading.
    faculty_heading = soup.find(
        lambda tag: tag.name in ("h1", "h2", "h3")
        and clean(tag.get_text()) == "Faculty"
    )

    if faculty_heading is None:
        raise ValueError("Could not find the Faculty section.")

    faculty_table = faculty_heading.find_next("table")
    if faculty_table is None:
        raise ValueError("Could not find the Faculty table.")

    records = []

    for row in faculty_table.find_all("tr")[1:]:  # skip header row
        cells = row.find_all(["td", "th"])

        # Faculty table columns: Name, Title, Email, Office, Research Interests
        if len(cells) < 2:
            continue

        name = clean(cells[0].get_text(" ", strip=True))
        title = clean(cells[1].get_text(" ", strip=True))

        if name and title:
            records.append({
                "Name": first_last_name(name),
                "Department": DEPARTMENT,
                "Title": title,
                "School": SCHOOL,
            })

    return pd.DataFrame(records)


faculty_df = scrape_csse_faculty()

print(faculty_df)
print(f"\nFaculty found: {len(faculty_df)}")

faculty_df.to_parquet("calpoly_csse_faculty.parquet", index=False)

                       Name                                 Department  \
0         Laila Abdelhafeez  Computer Science and Software Engineering   
1             Paul Anderson  Computer Science and Software Engineering   
2              Hisham Assal  Computer Science and Software Engineering   
3             Stephen Beard  Computer Science and Software Engineering   
4             John Bellardo  Computer Science and Software Engineering   
5              Alex Bisberg  Computer Science and Software Engineering   
6            Rodrigo Canaan  Computer Science and Software Engineering   
7             John Clements  Computer Science and Software Engineering   
8             Bruce DeBruhl  Computer Science and Software Engineering   
9             Alex Dekhtyar  Computer Science and Software Engineering   
10               Kirk Duran  Computer Science and Software Engineering   
11       Christian Eckhardt  Computer Science and Software Engineering   
12          Sussan Einakian  Computer 

CSU San Bernadino, Pomona, Humboldt, Somona


Schema:

Name (First Last)
School
Department
Title
Major (null if they don't have)
Sources: San Bernadino:

https://www.csusb.edu/campus-directory There is a dropdown to search by department, go through each department option and scrape all of the entries.
returns a table with firstname, lastname, department, and title
Pomona:

https://www.cpp.edu/faculty/index-new.shtml (should be 862 entries)
Each entry has a webpage like https://www.cpp.edu/faculty/{name}/index.shtml which has their title and department in this format Name new line, title newline, department, college
Humboldt:

Directory is a search page: https://www.humboldt.edu/its/directory?search=
results have a table with name, department, title. IF the title field is blank don’t scrape that entry
Somona:

https://ldaps.sonoma.edu/fasd/ has a drop down and query for each department, returns a table with name, department, and title
clean title ‘Instr Fac AY’ to Instructional Faculty
Run the four CSU collectors

Run the setup cell, then each campus cell below. All scraping, parsing, cleaning, and export code is contained in this notebook; no helper script is required.

Outputs are written to Faculty/Data/: csusb_faculty.parquet, cpp_faculty.parquet, humboldt_faculty.parquet, and sonoma_faculty.parquet.

All files use lowercase columns name, school, department, title, major, matching the other scraping notebook. Names are in first-last order. School is the university; department is the academic/administrative unit. Major is null unless explicitly supplied by the directory; it is not inferred from department or biography. Per the source notes, directory staff are retained. Humboldt entries without titles are omitted. Different appointments for the same person are retained; identical rows are deduplicated.

Successful HTTP responses are cached for resumable runs. Set REFRESH = True for a fresh scrape. Each campus writes a *_scrape_report.json with counts and errors. Failed department/search requests stop that campus export rather than replacing an existing file with incomplete data. Unavailable Pomona profiles retain the feed entry and are flagged in the report.

In [ ]:
import hashlib
import json
import logging
import re
import string
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlencode, urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# Resolve Faculty/Data whether Jupyter starts in data_scrapers, Faculty,
# CSC4693, or the DataEng workspace.
SCRAPER_DIR = next(
    (candidate for base in (Path.cwd(), *Path.cwd().parents)
     for candidate in (base, base / "data_scrapers", base / "Faculty" / "data_scrapers",
                       base / "CSC4693" / "Faculty" / "data_scrapers")
     if (candidate / "2_csu_data_scraper.ipynb").is_file()),
    None,
)
if SCRAPER_DIR is None:
    raise FileNotFoundError("Open this notebook from within the DataEng project.")
OUTPUT_DIR = SCRAPER_DIR.parent / "Data"
CACHE_DIR = SCRAPER_DIR / ".csu_cache" / "responses"
REFRESH = False  # True ignores cached responses and downloads fresh pages.
REQUEST_DELAY = 0.4
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

In [ ]:
URLS = {
    "csusb": "https://www.csusb.edu/campus-directory",
    "cpp": "https://www.cpp.edu/faculty/index-new.shtml",
    "humboldt": "https://www.humboldt.edu/its/directory?search=",
    "sonoma": "https://ldaps.sonoma.edu/fasd/",
}
SCHOOLS = {
    "csusb": "CSU San Bernardino", "cpp": "Cal Poly Pomona",
    "humboldt": "Cal Poly Humboldt", "sonoma": "Sonoma State University",
}
COLUMNS = ["name", "school", "department", "title", "major"]
LOG = logging.getLogger(__name__)


def clean(value):
    return re.sub(r"\s+", " ", str(value or "")).strip()


def first_last(value):
    """Reorder directory 'Last, First' names, keeping compound surnames."""
    value = clean(value)
    if "," in value:
        last, first = value.split(",", 1)
        value = f"{first.strip()} {last.strip()}"
    return clean(value)


def record(campus, name, department=None, title=None, major=None):
    return dict(zip(COLUMNS, [clean(name), SCHOOLS[campus], clean(department) or None,
                              clean(title) or None, clean(major) or None]))


class Client:
    """Cache successful responses for resumable runs; throttle and retry requests."""

    def __init__(self, cache_dir, delay=0.4, refresh=False):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.delay, self.refresh, self.last_request = max(0, delay), refresh, 0.0
        self.session = requests.Session()
        self.session.headers["User-Agent"] = "CSUFacultyCollector/1.0 (public academic directory research)"
        retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
                      allowed_methods=["GET", "POST"], respect_retry_after_header=True)
        self.session.mount("https://", HTTPAdapter(max_retries=retry))

    def get(self, url, *, data=None):
        method = "POST" if data is not None else "GET"
        key = hashlib.sha256(json.dumps([method, url, data], sort_keys=True).encode()).hexdigest()
        cached = self.cache_dir / f"{key}.html"
        if cached.exists() and not self.refresh:
            return cached.read_text(encoding="utf-8")
        time.sleep(max(0, self.delay - (time.monotonic() - self.last_request)))
        self.last_request = time.monotonic()
        response = self.session.request(method, url, data=data, timeout=(15, 60))
        response.raise_for_status()
        # These sites serve UTF-8 even when their HTTP headers omit charset.
        html = response.content.decode("utf-8", errors="replace")
        cached.write_text(html, encoding="utf-8")
        return html

    def close(self):
        self.session.close()


def soup(html):
    return BeautifulSoup(html, "html.parser")


def table_records(html, campus):
    """Match headers, not column positions (CSUSB has nine columns)."""
    doc = soup(html)
    records, recognized = [], False
    for table in doc.select("table"):
        caption = table.find("caption")
        if campus == "humboldt" and caption and "Personnel Listings" not in caption.get_text():
            continue
        headings = table.select("tr th")
        headers = [clean(h.get_text(" ", strip=True)).lower() for h in headings]
        if not {"department", "title"}.issubset(headers):
            continue
        if "name" not in headers and not {"first name", "last name"}.issubset(headers):
            continue
        recognized = True
        for tr in table.select("tr"):
            cells = tr.find_all("td", recursive=False)
            if not cells:
                continue
            if len(cells) != len(headers):
                raise ValueError(f"{campus}: malformed directory row ({len(cells)} cells, {len(headers)} headers)")
            values = dict(zip(headers, [clean(td.get_text(" ", strip=True)) for td in cells]))
            name = (clean(f"{values['first name']} {values['last name']}") if "first name" in values
                    else first_last(values["name"]))
            title = values["title"]
            if campus == "humboldt" and not title:
                continue
            if campus == "sonoma":
                title = re.sub(r"\bInstr\s+Fac\s+AY\b", "Instructional Faculty", title, flags=re.I)
            if name:
                records.append(record(campus, name, values["department"], title, values.get("major")))
    if not recognized:
        text = doc.get_text(" ", strip=True).lower()
        if not any(message in text for message in ("no results", "no matches", "no people", "no records")):
            raise ValueError(f"{campus}: no recognizable directory table or explicit empty-results message")
    return records


def next_page(html, url):
    doc = soup(html)
    link = doc.select_one('a[rel~="next"], .pager__item--next a')
    if not link:
        return None
    target = urljoin(url, link["href"])
    if urlparse(target).netloc != urlparse(url).netloc:
        raise ValueError(f"Unexpected external pagination link: {target}")
    return target


def read_pages(client, url, campus, *, data=None):
    records, visited = [], set()
    while url:
        if url in visited:
            raise ValueError(f"Pagination loop at {url}")
        visited.add(url)
        html = client.get(url, data=data)
        records.extend(table_records(html, campus))
        url, data = next_page(html, url), None
    return records


def department_options(html, selector):
    select = soup(html).select_one(selector)
    if select is None:
        raise ValueError(f"Department dropdown missing: {selector}")
    options = [(option.get("value", ""), clean(option.get_text())) for option in select.select("option")]
    options = [(value, label) for value, label in options if value and value.lower() != "all"]
    if not options:
        raise ValueError("Department dropdown contains no departments")
    return options


def collect_departments(client, campus, report):
    url = URLS[campus]
    landing = client.get(url)
    field = "field_department_titles" if campus == "csusb" else "dept"
    options = department_options(landing, f'select[name="{field}"]')
    report["departments_total"] = len(options)
    rows = []
    for index, (value, label) in enumerate(options, 1):
        if campus == "csusb":
            target, data = url + "?" + urlencode({field: value}), None
        else:
            form = soup(landing).select_one('select[name="dept"]').find_parent("form")
            target = urljoin(url, form.get("action", ""))
            data = {i["name"]: i.get("value", "") for i in form.select('input[type="hidden"][name]')}
            data[field] = value
        try:
            found = read_pages(client, target, campus, data=data)
            rows.extend(found)
            report["departments"].append({"id": value, "name": label, "rows": len(found)})
        except (requests.RequestException, ValueError) as error:
            report["errors"].append({"url": target, "department": label, "error": str(error)})
            LOG.error("%s / %s: %s", campus, label, error)
        if index % 25 == 0 or index == len(options):
            LOG.info("%s: %d/%d departments, %d rows", campus, index, len(options), len(rows))
    if report["errors"]:
        raise RuntimeError(f"{campus}: {len(report['errors'])} department requests failed; output not replaced")
    return rows


def collect_campus(campus, output_dir=OUTPUT_DIR, *, delay=0.4, refresh=False,
                   cache_dir=CACHE_DIR):
    """Collect and atomically export a campus; also write an audit JSON report.

    Successful cached pages are reused until refresh=True. A failed department
    or search does not replace existing data. Failed Pomona profiles retain the
    feed row and are explicitly listed in the report, with status 'partial'.
    """
    if campus not in URLS:
        raise ValueError(f"Unknown campus: {campus}")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    destination = output_dir / f"{campus}_faculty.parquet"
    report = {"campus": campus, "source_url": URLS[campus], "started_at": datetime.now(timezone.utc).isoformat(),
              "status": "running", "departments": [], "searches": [], "errors": []}
    client = Client(cache_dir, delay, refresh)
    try:
        if campus in {"csusb", "sonoma"}:
            rows = collect_departments(client, campus, report)
        elif campus == "cpp":
            rows = collect_cpp(client, report)
        else:
            rows = collect_humboldt(client, report)
        if not rows:
            raise ValueError(f"{campus}: zero records; refusing to write an empty file")
        frame = pd.DataFrame(rows, columns=COLUMNS).astype("string")
        frame = frame.drop_duplicates().sort_values(["name", "department", "title"], na_position="last").reset_index(drop=True)
        temporary = destination.with_suffix(".parquet.tmp")
        frame.to_parquet(temporary, index=False, engine="pyarrow")
        temporary.replace(destination)
        report.update(status="partial" if report["errors"] else "complete", rows=len(frame),
                      unique_names=frame["name"].nunique(), missing_titles=int(frame["title"].isna().sum()),
                      output=str(destination))
        LOG.info("Wrote %d rows to %s (status: %s)", len(frame), destination, report["status"])
        return frame
    except Exception as error:
        report.update(status="failed", error=str(error))
        raise
    finally:
        client.close()
        report["finished_at"] = datetime.now(timezone.utc).isoformat()
        (output_dir / f"{campus}_scrape_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

CSU San Bernadino

In [ ]:
csusb_df = collect_campus("csusb", OUTPUT_DIR, delay=REQUEST_DELAY, refresh=REFRESH)
print(f"Saved {len(csusb_df):,} rows to {OUTPUT_DIR / 'csusb_faculty.parquet'}")
csusb_df.head()

Cal Poly Pomona

In [8]:
def parse_cpp_profile(html, entry):
    doc = soup(html)
    info = doc.select_one(".fac-info")
    if info is None or info.select_one("h1.profheader") is None:
        raise ValueError("Faculty profile header missing")
    title = info.select_one("p.title")
    affiliation = info.select_one("p.college")
    department = clean(entry.get("school"))  # Feed's 'school' is the department, 'dept' the college.
    if affiliation:
        text = clean(affiliation.get_text(" ", strip=True))
        college = clean(entry.get("dept"))
        if college and text.endswith(college):
            department = text[:-len(college)].rstrip(" ,") or department
        elif re.search(r",\s*(?:College of|[\w .'-]+ College\b)", text):
            department = re.split(r",\s*(?=College of|[\w .'-]+ College\b)", text, maxsplit=1)[0]
        elif text:
            department = text
    major = info.select_one(".major")
    return record("cpp", clean(info.select_one("h1.profheader").get_text(" ", strip=True)),
                  department, title.get_text(" ", strip=True) if title else entry.get("title"),
                  major.get_text(" ", strip=True) if major else None)


def collect_cpp(client, report):
    landing = client.get(URLS["cpp"])
    match = re.search(r'var\s+jsonFeed\s*=\s*[\"\']([^\"\']+)', landing)
    if not match:
        raise ValueError("Pomona directory JSON feed link missing")
    feed_url = urljoin(URLS["cpp"], match.group(1))
    entries = json.loads(client.get(feed_url))["faculty"]
    if not entries:
        raise ValueError("Pomona directory feed is empty")
    report["directory_entries"] = len(entries)
    report["expected_entries_from_notes"] = 862
    rows = []
    for index, entry in enumerate(entries, 1):
        url = urljoin(URLS["cpp"], entry.get("link", ""))
        try:
            if urlparse(url).hostname not in {"www.cpp.edu", "cpp.edu"} or not entry.get("link"):
                raise ValueError("Missing or external faculty profile link")
            rows.append(parse_cpp_profile(client.get(url), entry))
        except (requests.RequestException, ValueError) as error:
            # Keep the directory-listed person even if their profile is unavailable.
            rows.append(record("cpp", first_last(entry["name"]), entry.get("school"), entry.get("title")))
            report["errors"].append({"url": url, "name": entry["name"], "error": str(error)})
            LOG.warning("Pomona profile unavailable; using feed fields: %s (%s)", url, error)
        if index % 50 == 0 or index == len(entries):
            LOG.info("cpp: %d/%d profiles", index, len(entries))
    return rows

cpp_df = collect_campus("cpp", OUTPUT_DIR, delay=REQUEST_DELAY, refresh=REFRESH)
print(f"Saved {len(cpp_df):,} rows to {OUTPUT_DIR / 'cpp_faculty.parquet'}")
cpp_df.head()



Cal Poly Humboldt

In [9]:
def collect_humboldt(client, report):
    # The live server accepts one-letter searches despite its three-letter UI tip.
    # Searching every letter covers names/titles without requiring an email/phone.
    # Include domain/phone searches as additional coverage for non-Latin names.
    rows = []
    for term in [*string.ascii_lowercase, "humboldt.edu", "826"]:
        url = URLS["humboldt"].split("?", 1)[0] + "?" + urlencode({"search": term})
        found = read_pages(client, url, "humboldt")
        report["searches"].append({"term": term, "rows": len(found)})
        rows.extend(found)
        LOG.info("humboldt: search %r, %d rows with titles", term, len(found))
    return rows

humboldt_df = collect_campus("humboldt", OUTPUT_DIR, delay=REQUEST_DELAY, refresh=REFRESH)
print(f"Saved {len(humboldt_df):,} rows to {OUTPUT_DIR / 'humboldt_faculty.parquet'}")
humboldt_df.head()

Sonoma State University

In [ ]:
sonoma_df = collect_campus("sonoma", OUTPUT_DIR, delay=REQUEST_DELAY, refresh=REFRESH)
print(f"Saved {len(sonoma_df):,} rows to {OUTPUT_DIR / 'sonoma_faculty.parquet'}")
sonoma_df.head()

CSU San Marcos

In [ ]:
import re, sys

import polars as pl
import requests
from bs4 import BeautifulSoup

URL = "https://catalog.csusm.edu/content.php?catoid=12&navoid=2267"
SCHOOL = "California State University San Marcos"
EXTRA = r",? (?:Jr\.?|Sr\.?|II|III|IV)$|\s*\(.*?\)"

html = open(sys.argv[1], encoding="utf-8").read() if len(sys.argv) > 1 else \
    requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
main = BeautifulSoup(html, "html.parser").find("td", class_="block_content")
for br in main.find_all("br"):
    br.replace_with("|")

rows, current = [], False
for el in main.find_all(["h2", "h3", "p"]):
    text = re.sub(r"\s+", " ", el.get_text()).strip()
    if el.name != "p":
        current = text == "Academic and Administrative Listing"
        continue
    lines = [s.strip() for s in text.split("|") if s.strip()]
    if not current or len(lines) < 2 or not el.find("strong") or "," not in lines[0]:
        continue
    nm = re.match(r"(.+?)\s*(?:\(\w+\))?$", lines[0]).group(1)
    last, first = [re.sub(EXTRA, "", s.strip()).strip() for s in nm.split(",", 1)]
    title, _, dept = [s.strip() for s in lines[1].partition(",")]
    dept = re.sub(r"^Department Chair, ", "", dept)
    rows.append(dict(name=f"{first} {last}", school=SCHOOL, department=dept or None, title=title, major=None))

df = pl.DataFrame(rows, schema_overrides={"major": pl.Utf8}).unique(subset=["name"], maintain_order=True)
(df.filter(pl.col("title").str.contains("Professor|Librarian"))
   .with_columns(pl.col("name").str.replace_all(r" (?:[A-Za-z]\.? )+", " "))
   .select("name", "school", "department", "title", "major")
   .write_parquet("csusm_cleaned.parquet", compression="zstd"))

CSU Fullerton

In [ ]:
import re, sys

import polars as pl
import requests
from bs4 import BeautifulSoup

URL = "https://catalog.fullerton.edu/content.php?catoid=101&navoid=15092"
SCHOOL = "California State University, Fullerton"
RANK = r"((?:Assist\w+ |Assoc\w+ )?Professor|Lecturer|(?:Senior Assistant |Associate )?Librarian)"
EXTRA = r",? (?:Jr\.?|Sr\.?|II|III|IV)$|\s*\(.*?\)"
DEGREE = r"\s+(?=(?:[A-Z][a-z]?\.){2,}|Ph\.D|Ed\.D|MBA\b)"

html = open(sys.argv[1], encoding="utf-8").read() if len(sys.argv) > 1 else \
    requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
main = BeautifulSoup(html, "html.parser").find("td", class_="block_content")
for br in main.find_all("br"):
    br.replace_with("|")

rows = []
for p in main.find_all("p"):
    lines = [s.strip() for s in re.sub(r"\s+", " ", p.get_text().replace("​", "")).split("|")]
    head = lines[0]
    m = re.match(r"(.+?)\s*\(\s*(\d{4})\s*\)\s*,?\s*(.*)", head)
    if not m and re.search(RANK, head):
        m = re.match(r"([^,]+,[^,]+?)()\s*,\s*(.*)", head)
    if not m:
        continue
    nm, _, raw = m.groups()
    if len(lines) == 1:
        raw = re.split(DEGREE, raw)[0]
    last, first = [re.sub(EXTRA, "", s.strip(" ,")).strip() for s in (nm.split(",", 1) if "," in nm else nm.split(" ", 1))]

    seg = re.split(r"[;:]", raw)[0].strip(" ,")
    m = re.match(rf"{RANK}(?: (?:of|in) (.+))?$", seg)
    title, dept = m.groups() if m else (seg.partition(", ")[0], seg.partition(", ")[2])
    title = re.sub(r"Assoc\w+ P", "Associate P", re.sub(r"Assist\w+ P", "Assistant P", title))
    rows.append(dict(name=f"{first} {last}", school=SCHOOL, department=dept or None, title=title or None,
                     major=None, raw=raw or None))

df = pl.DataFrame(rows, schema_overrides={"major": pl.Utf8}).unique(subset=["name", "raw"], maintain_order=True)
(df.filter(pl.col("title").str.contains("Professor|Librarian") | (pl.col("title") == "Lecturer"))
   .with_columns(pl.col("name").str.replace_all(r" (?:[A-Za-z]\.? )+", " "))
   .select("name", "school", "department", "title", "major")
   .write_parquet("fullerton_cleaned.parquet", compression="zstd"))

San Diego State University

In [ ]:
import re, sys

import polars as pl
import requests
from bs4 import BeautifulSoup

URL = "https://caa.sdsu.edu/curriculum/faculty-listing"
SCHOOL = "San Diego State University"
SECTIONS = {"Tenured": "tenure_track", "Additional": "tenure_track", "Lecturers": "lecturer",
            "Adjunct": "adjunct", "Emeritus": "emeritus"}
DEGREE = r"(?:[A-Z][A-Za-z]{0,5}\.){1,4}[A-Za-z]{0,3}\.?|Doctor of\.?|Ph\.? ?D\.?|MFA|MBA|MA|MS|JD|DPT|DNP"
ADMIN = r"Director|President|Dean|Chair|Coordinator|Provost|Program|Advis|Professor"
RANK = r"((?:Distinguished |Assistant |Associate )?(?:Professor|Librarian)|Lecturer|Counselor)"

html = open(sys.argv[1], encoding="utf-8").read() if len(sys.argv) > 1 else \
    requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
main = BeautifulSoup(html, "html.parser").find("main")

rows, cat = [], None
for el in main.find_all(["h2", "li"]):
    text = re.sub(r"\s+", " ", el.get_text(" ", strip=True).replace("​", "")).replace(" ,", ",")
    if el.name == "h2":
        cat = next((v for k, v in SECTIONS.items() if text.startswith(k)), None)
        continue
    if cat is None or text.count(",") < 1:
        continue
    last, first, *rest = [p.strip() for p in text.split(",", 2)]
    rest = rest[0] if rest else ""
    if re.fullmatch(r"II|III|IV|Jr\.?|Sr\.?", first):
        first, _, rest = (p.strip() for p in rest.partition(","))
    if re.match(RANK, first) and " " in last:
        rest, (first, last) = f"{first}, {rest}", last.rsplit(" ", 1)
    m = re.search(rf"(?:(?<=[a-z])\.)? (?={RANK}|Associate Dean)", first)
    if m:
        first, rest = first[:m.start()], f"{first[m.end():]}, {rest}"
    first = re.sub(r"^(?:MSG|LTC|CPT|Capt|Maj|Col|SFC|FNU) |\s*\(.*", "", first).strip()
    rest = re.split(r"\s*\((?=[A-Z])|\s(?=B\.[AS]\.,)", rest)[0]
    rest = re.sub(r",? SDSU Imperial Valley", "", rest)

    title, dept = None, None
    if cat in ("lecturer", "adjunct"):
        dept = re.sub(rf"^(?:(?:{DEGREE})[,;/ ]*)+", "", rest).strip(" ,.;(") or None
        title = "Lecturer" if cat == "lecturer" else "Adjunct"
    else:
        rest = re.sub(r"^(?:[A-Z][A-Za-z]*\.?[A-Z]?\.?, )?\d{4}-\d{4}, ", "", rest)
        m = re.search(rf"{RANK}s?(?: (?:of|in|,) (.+))?", rest)
        if m:
            title = m.group(1)
            dept = re.split(r",? and (?:Interim |Assistant |Associate |Vice |Director|Chair|Dean|Professor)|"
                            r"; |, (?:The|Director|Chair|Dean|Emerit)", m.group(2) or "")[0].strip(" ,.;(") or None
            if dept:
                parts = re.sub(r"\s*\[.*?\]", "", dept).split(", ")
                cut = next((i for i, p in enumerate(parts) if re.search(ADMIN, p)), len(parts))
                dept = ", ".join(parts[:cut]).strip(" ,.;(") or None
    rows.append(dict(name=f"{first} {last}", school=SCHOOL, department=dept, title=title, major=None,
                     category="librarian" if cat == "tenure_track" and "Librarian" in (title or "") else cat))

df = pl.DataFrame(rows, schema_overrides={"major": pl.Utf8}).unique(subset=["name", "category"], keep="first", maintain_order=True)
(df.filter(pl.col("category").is_in(["tenure_track", "lecturer", "adjunct", "librarian"]) & pl.col("title").is_not_null())
   .with_columns(pl.col("name").str.replace_all(r" (?:[A-Za-z]\.? )+", " "))
   .select("name", "school", "department", "title", "major")
   .write_parquet("sdsu_cleaned.parquet", compression="zstd"))

Combining all CSU data

In [26]:
import pandas as pd
import duckdb
import numpy as np
from pathlib import Path

#read every parquet in ../Data
data_dir = Path("/content/Data").resolve()
output_path = data_dir / "csu_combined.parquet"

parquet_files = sorted(
    path for path in data_dir.glob("*.parquet")
    if path != output_path
)

if not parquet_files:
    raise FileNotFoundError(f"No input parquet files found in {data_dir}")

combined = pd.concat(
    [pd.read_parquet(path) for path in parquet_files],
    axis=0,
    join="outer",
    ignore_index=True,
    sort=False,
)

combined.to_parquet(output_path, index=False)

print(f"Combined {len(parquet_files)} files: {combined.shape[0]:,} rows, "
      f"{combined.shape[1]} columns")
print(f"Saved to: {output_path}")
print(combined.shape)
combined.head(10)

Combined 22 files: 17,313 rows, 7 columns
Saved to: /content/Data/csu_combined.parquet
(17313, 7)


,name,title,department,school,year,major,start_date
0,Dawn Abel,Lecturer,Nursing,"California State University, Chico",2013,NaN,NaN
1,Annie Adamian,Associate Professor,Education,"California State University, Chico",2014,NaN,NaN
2,Nora Aguilar McKay,Lecturer,Education,"California State University, Chico",2014,NaN,NaN
3,Kelsey Aguire,Lecturer,Nursing,"California State University, Chico",2025,NaN,NaN
4,Kirk Aiken,Professor,Marketing,"California State University, Chico",2013,NaN,NaN
5,Hannah Aird,Associate Professor,Earth & Environmental Sci,"California State University, Chico",2015,NaN,NaN
6,Oleksiy Al-Saadi,Assistant Professor,Computer Science,"California State University, Chico",2025,NaN,NaN
7,Zahrasadat Alavi,Associate Professor,Electrical & Computer Engr,"California State University, Chico",2017,NaN,NaN
8,Mohammed Albahttiti,Associate Professor,Concrete Industry Mgmt,"California State University, Chico",2017,NaN,NaN
9,Kelli Albertson,Lecturer,Physics; Science Education,"California State University, Chico",2008,NaN,NaN
